# Hierarchical IMSTE quality check

Compare the hierarchical graph and embedding with exact IMSTE on the built-in handwritten digits dataset. The sweep varies `n_clusters` and reports graph-edge recall against the exact graph, trustworthiness, post-hoc 5-NN accuracy, and fit time. The label-based score is for evaluation only; labels are not passed to either embedder. Exact-edge recall is a reference measure, while trustworthiness and 5-NN accuracy offer complementary views of embedding quality.

For a quicker run, reduce `MAX_SAMPLES`, `N_EPOCHS`, or `N_CLUSTERS_TO_TRY`.

In [ ]:
import time
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.manifold import trustworthiness
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'mst_embedding').is_dir()),
    None,
)
if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from mst_embedding import IteratedMinimumSpanningTreeEmbedder

SEED = 42
MAX_SAMPLES = 1200  # Set to None to use all 1,797 observations.
N_MSTS = 3
N_EPOCHS = 250
N_CLUSTERS_TO_TRY = [10, 25, 50, 100]
N_COARSE_MSTS = 2
N_LOCAL_MSTS = 3
MINIBATCH_SIZE = 256
N_JOBS = 2

digits = load_digits()
X = StandardScaler().fit_transform(digits.data)
y = digits.target
if MAX_SAMPLES is not None and MAX_SAMPLES < len(X):
    selected, _ = train_test_split(
        np.arange(len(X)), train_size=MAX_SAMPLES, random_state=SEED, stratify=y
    )
    X, y = X[selected], y[selected]
print(f'Using {len(X):,} samples with {X.shape[1]} standardized features.')

## Exact reference

Fit exact IMSTE once. Its graph supplies the edge-recovery reference for every hierarchical run.

In [ ]:
exact_model = IteratedMinimumSpanningTreeEmbedder(
    graph_mode='exact',
    n_msts=N_MSTS,
    n_epochs=N_EPOCHS,
    batch_size=4096,
    random_state=SEED,
    device='cpu',
).fit(X)
exact_edges = {tuple(sorted(map(int, edge))) for edge in exact_model.graph_edges_}
print(f'Exact graph: {len(exact_edges):,} unique edges')

## Hierarchical sweep

Each run uses the same seed and embedding settings. Only the cluster count changes. `n_coarse_msts` controls how many centroid-tree layers select cluster pairs; `n_local_msts` controls the tree ranks constructed over each pair's original samples.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rows = []
hierarchical_models = {}

for n_clusters in N_CLUSTERS_TO_TRY:
    started = time.perf_counter()
    model = IteratedMinimumSpanningTreeEmbedder(
        graph_mode='hierarchical',
        n_clusters=n_clusters,
        n_coarse_msts=N_COARSE_MSTS,
        n_local_msts=N_LOCAL_MSTS,
        n_jobs=N_JOBS,
        minibatch_size=MINIBATCH_SIZE,
        n_epochs=N_EPOCHS,
        batch_size=4096,
        random_state=SEED,
        device='cpu',
    ).fit(X)
    fit_seconds = time.perf_counter() - started
    hierarchical_models[n_clusters] = model
    graph_edges = {tuple(sorted(map(int, edge))) for edge in model.graph_edges_}
    recovered = len(graph_edges & exact_edges)
    edge_recall = recovered / max(len(exact_edges), 1)
    edge_precision = recovered / max(len(graph_edges), 1)
    tw = trustworthiness(X, model.embedding_, n_neighbors=10)
    knn_accuracy = cross_val_score(
        KNeighborsClassifier(n_neighbors=5),
        model.embedding_, y, cv=cv, n_jobs=N_JOBS,
    ).mean()
    rows.append({
        'n_clusters': n_clusters,
        'unique_edges': len(graph_edges),
        'exact_edge_recall': edge_recall,
        'exact_edge_precision': edge_precision,
        'trustworthiness': tw,
        '5NN_accuracy': knn_accuracy,
        'fit_seconds': fit_seconds,
    })
    print(f'{n_clusters} clusters: {fit_seconds:.1f}s, recall={edge_recall:.3f}, trust={tw:.3f}')

results = pd.DataFrame(rows).sort_values('n_clusters').reset_index(drop=True)
results

## Exact embedding reference

Compute the same quality measures for the exact embedding. Scores are diagnostics, not training objectives.

In [ ]:
exact_tw = trustworthiness(X, exact_model.embedding_, n_neighbors=10)
exact_knn = cross_val_score(
    KNeighborsClassifier(n_neighbors=5), exact_model.embedding_, y, cv=cv, n_jobs=N_JOBS
).mean()
exact_summary = pd.DataFrame([{
    'method': 'exact',
    'unique_edges': len(exact_edges),
    'trustworthiness': exact_tw,
    '5NN_accuracy': exact_knn,
}])
exact_summary

## Compare metrics and embeddings

Higher edge recall means more exact-graph edges were recovered. Trustworthiness measures neighborhood preservation from the original feature space, and 5-NN accuracy is a label-based post-fit diagnostic. Compare the full pattern rather than treating any one score as definitive.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8), constrained_layout=True)
for ax, metric, title in zip(
    axes,
    ['exact_edge_recall', 'trustworthiness', '5NN_accuracy', 'fit_seconds'],
    ['Exact edge recall', 'Trustworthiness', '5-NN accuracy', 'Fit time (s)'],
):
    ax.plot(results['n_clusters'], results[metric], marker='o')
    ax.set_xlabel('Number of clusters')
    ax.set_title(title)
    ax.grid(alpha=0.25)
axes[0].set_ylim(0, 1)
axes[1].set_ylim(0, 1)
axes[2].set_ylim(0, 1)
plt.show()

ncols = min(3, len(hierarchical_models) + 1)
nrows = int(np.ceil((len(hierarchical_models) + 1) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False, constrained_layout=True)
panels = [('Exact', exact_model.embedding_)] + [
    (f'Hierarchical, k={k}', model.embedding_)
    for k, model in hierarchical_models.items()
]
for ax, (title, embedding) in zip(axes.flat, panels):
    points = ax.scatter(embedding[:, 0], embedding[:, 1], c=y, cmap='tab10', s=8, alpha=0.8)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
for ax in axes.flat[len(panels):]:
    ax.set_visible(False)
fig.colorbar(points, ax=list(axes.flat[:len(panels)]), label='Digit')
plt.show()